# Model Architecture

The model is built by `src/model.py` (`build_unet`) from the `segmentation-models-pytorch` library.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
while not (ROOT / "config.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import torch
import segmentation_models_pytorch as smp

import config
from src.model import build_unet

# Architecture only: instantiate with random encoder weights to avoid the ImageNet
# download. The layer structure is identical to the pretrained model used in training
# (encoder_weights set the parameter values, not the architecture).
model = build_unet(encoder_weights=None).eval()
print("segmentation-models-pytorch", smp.__version__, "| torch", torch.__version__)
print(f"encoder: {config.ENCODER_NAME}  |  weights (training): {config.ENCODER_WEIGHTS}")
print("in_channels: 3 (RGB)  |  classes: 1 (EI)  |  output activation: sigmoid")

## Components and parameter counts

The U-Net is assembled from three parts: the ResNet-34 encoder (contracting path), the U-Net decoder (expanding path with skip connections), and the segmentation head (final convolution + sigmoid).

In [ ]:
def count(module):
    """Total number of parameters in a module."""
    return sum(p.numel() for p in module.parameters())

print(f"{'component':<22}{'parameters':>14}")
print(f"{'encoder (resnet34)':<22}{count(model.encoder):>14,}")
print(f"{'decoder (U-Net)':<22}{count(model.decoder):>14,}")
print(f"{'segmentation head':<22}{count(model.segmentation_head):>14,}")
print(f"{'total':<22}{count(model):>14,}")

## Feature-map shapes from input to output

A 256×256 RGB crop is passed through the full network to trace both paths. In the **encoder** (contracting path) the spatial resolution is progressively halved while the channel depth increases. In the **decoder** (expanding path) the resolution is restored back to full size while the channel depth shrinks, each block combining the upsampled features with the matching encoder skip connection. The sigmoid head then returns a single-channel EI map in [0, 1]. Decoder shapes are captured with forward hooks on the decoder blocks.

In [ ]:
x = torch.randn(1, 3, config.CROP_SIZE, config.CROP_SIZE)

# Forward hooks capture each decoder block's output shape during the full forward pass.
dec_shapes = []
hooks = [blk.register_forward_hook(lambda mod, inp, out: dec_shapes.append(tuple(out.shape)))
         for blk in model.decoder.blocks]
with torch.no_grad():
    feats = model.encoder(x)          # one feature map per encoder stage
    out = model(x)                    # full forward also fills dec_shapes
for h in hooks:
    h.remove()

def row(name, shape):
    c, h, w = shape[1], shape[2], shape[3]
    return f"  {name:<12}{str((c, h, w)):<18}{'/' + str(config.CROP_SIZE // h):>10}"

print(f"input: {tuple(x.shape)}  (N, C, H, W)\n")
print(f"  {'stage':<12}{'(C, H, W)':<18}{'downsample':>10}")
print("ENCODER (contracting path) — resolution halves, channels grow:")
for i, f in enumerate(feats):
    print(row(f"stage {i}", tuple(f.shape)))
print("DECODER (expanding path) — resolution restored, channels shrink:")
for i, s in enumerate(dec_shapes):
    print(row(f"block {i}", s))
print(f"\nhead -> output: {tuple(out.shape)}  (sigmoid, values in "
      f"[{float(out.min()):.3f}, {float(out.max()):.3f}])")

## Full layer-by-layer structure

The complete module tree: the encoder's stem and four residual stages, the decoder's upsampling blocks (each concatenating the matching encoder skip connection), and the segmentation head.

In [ ]:
print(model)